In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

In [6]:
class NaiveBayesClassifier:
    def __init__(self, smoothing=1e-6):
        self.classes = None
        self.priors = {}
        self.likelihoods = {}
        self.smoothing = smoothing  # Variance smoothing parameter

    def fit(self, X, y):
        self.classes = np.unique(y)
        for cls in self.classes:
            X_cls = X[y == cls]
            self.priors[cls] = len(X_cls) / len(X)

            self.likelihoods[cls] = {
                feature: (
                    np.mean(X_cls[:, feature]),
                    np.std(X_cls[:, feature]) + self.smoothing,
                )
                for feature in range(X.shape[1])
            }

    def _gaussian_pdf(self, x, mean, std):
        pdf = (1 / (np.sqrt(2 * np.pi) * std)) * np.exp(
            -((x - mean) ** 2 / (2 * std**2))
        )
        return max(pdf, 1e-300)  # Clip probabilities to avoid numerical issues

    def log_likelihood(self, X, y):
        log_likelihood = 0
        for i, sample in enumerate(X):
            cls = y[i]
            prior = np.log(self.priors[cls])
            likelihood = sum(
                np.log(
                    max(
                        self._gaussian_pdf(
                            sample[feature], *self.likelihoods[cls][feature]
                        ),
                        1e-300,
                    )
                )
                for feature in range(len(sample))
            )
            log_likelihood += prior + likelihood
        return log_likelihood

    def predict(self, X):
        predictions = []
        for sample in X:
            posteriors = {}
            for cls in self.classes:
                prior = np.log(self.priors[cls])
                likelihood = sum(
                    np.log(
                        max(
                            self._gaussian_pdf(
                                sample[feature], *self.likelihoods[cls][feature]
                            ),
                            1e-300,
                        )
                    )
                    for feature in range(len(sample))
                )
                posteriors[cls] = prior + likelihood

            predictions.append(max(posteriors, key=posteriors.get))
        return np.array(predictions)

    def bic_score(self, X, y):
        n_samples, n_features = X.shape
        log_likelihood = self.log_likelihood(X, y)
        num_parameters = (
            len(self.classes) * n_features * 2
        )  # Mean and variance per feature per class
        return -2 * log_likelihood + num_parameters * np.log(n_samples)

In [3]:
df = pd.read_parquet("../../data/benchmark/testing/eth")

X = df.drop(columns=['label']).to_numpy()
y = df['label'].to_numpy()

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [8]:
# Train and compute BIC for multiple models
nb1 = NaiveBayesClassifier(smoothing=1e-6)
nb1.fit(X_train, y_train)
bic1 = nb1.bic_score(X_train, y_train)

nb2 = NaiveBayesClassifier(smoothing=1e-3)
nb2.fit(X_train, y_train)
bic2 = nb2.bic_score(X_train, y_train)

# Compare BIC scores
print(f"BIC (smoothing=1e-6): {bic1}")
print(f"BIC (smoothing=1e-3): {bic2}")

better_model = "nb1" if bic1 < bic2 else "nb2"
print(f"Better model (based on BIC): {better_model}")

BIC (smoothing=1e-6): 4296900.670406447
BIC (smoothing=1e-3): 14939781.550427385
Better model (based on BIC): nb1


In [9]:
y_pred = nb1.predict(X_test)

In [11]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.4183100767210751
Classification Report:
               precision    recall  f1-score   support

       False       0.98      0.26      0.42     15253
        True       0.27      0.98      0.42      4168

    accuracy                           0.42     19421
   macro avg       0.62      0.62      0.42     19421
weighted avg       0.83      0.42      0.42     19421

